# Chapter 10 Lab — Supervisor-Worker Pattern

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 10 — Supervisor-Worker Pattern  
**Goal:** Build a supervisor agent that decomposes complex tasks, delegates to specialized workers, runs them in parallel, and aggregates results — the backbone of production multi-agent systems.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | The Case for Hierarchy |
| 2 | Architecture Overview |
| 3 | Designing the Supervisor |
| 4 | Building Worker Agents |
| 5 | Task Decomposition |
| 6 | Parallel Execution |
| 7 | Result Aggregation |
| 8 | Error Handling |
| 9 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.  
> This lab uses **mock LLM responses** throughout so you can run every cell without API calls.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install langgraph langchain-openai python-dotenv --quiet

In [ ]:
import asyncio
import json
import os
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Callable

from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

print("All imports loaded successfully.")

---
## 1. The Case for Hierarchy

When you throw three or four agents into a flat group chat, coordination feels effortless. But at **eight or more agents**, things break down fast:

| Problem | Flat Multi-Agent | Supervisor-Worker |
|---------|-----------------|-------------------|
| **Token cost** | Every agent sees every message — O(n^2) tokens | Workers only see their own subtask |
| **Conflicting actions** | Agents step on each other | Supervisor serializes decisions |
| **Deadlocks** | Agent A waits for B, B waits for A | Supervisor breaks ties |
| **Accountability** | Who caused the bad output? | Supervisor logs every delegation |

Let's prove it. Below we simulate a flat system where 6 agents try to coordinate on a market-research task.

In [ ]:
# ── Flat multi-agent simulation ──────────────────────────────────────────────
# Six agents, each with a specialty, all sharing one message bus.

import random

FLAT_AGENTS = [
    "researcher", "analyst", "writer",
    "fact_checker", "editor", "formatter",
]

def simulate_flat_multi_agent(task: str, rounds: int = 5) -> dict:
    """Simulate flat coordination — every agent sees all messages."""
    shared_messages = [{"role": "user", "content": task}]
    total_tokens = 0
    conflicts = 0

    for round_num in range(rounds):
        acting_agents = random.sample(FLAT_AGENTS, k=random.randint(2, 4))
        round_outputs = []
        for agent in acting_agents:
            # Each agent "reads" all shared messages — token cost grows
            tokens_this_call = len(shared_messages) * 150  # ~150 tokens/msg
            total_tokens += tokens_this_call
            output = f"[{agent}] Round {round_num+1}: I think we should..."
            round_outputs.append(output)

        # Detect conflicts: multiple agents editing the same section
        if "writer" in acting_agents and "editor" in acting_agents:
            conflicts += 1
        if "researcher" in acting_agents and "fact_checker" in acting_agents:
            conflicts += 1

        shared_messages.extend(
            {"role": "assistant", "content": o} for o in round_outputs
        )

    return {
        "total_messages": len(shared_messages),
        "estimated_tokens": total_tokens,
        "conflicts": conflicts,
        "rounds": rounds,
    }


result = simulate_flat_multi_agent("Write a market research report on EV adoption")
print("Flat Multi-Agent Simulation")
print("=" * 40)
for k, v in result.items():
    print(f"  {k:20s}: {v}")
print(f"\nConflicts detected: {result['conflicts']}")
print("Token cost scales quadratically as more agents join the conversation.")

In [ ]:
# ── Compare: hierarchical approach (preview) ────────────────────────────────
# With a supervisor, each worker gets ONLY its subtask — no shared bus.

def simulate_supervisor_approach(task: str, num_workers: int = 4) -> dict:
    """Supervisor sends isolated subtasks to workers."""
    supervisor_tokens = 300  # reads task, decides plan
    worker_tokens = num_workers * 200  # each worker reads only its subtask
    aggregation_tokens = num_workers * 100  # supervisor reads summaries
    total_tokens = supervisor_tokens + worker_tokens + aggregation_tokens
    return {
        "total_messages": 1 + num_workers + num_workers + 1,  # task + delegations + results + final
        "estimated_tokens": total_tokens,
        "conflicts": 0,  # supervisor prevents conflicts by design
    }


hier = simulate_supervisor_approach("Write a market research report on EV adoption")
print("Supervisor-Worker Simulation")
print("=" * 40)
for k, v in hier.items():
    print(f"  {k:20s}: {v}")

print(f"\nToken savings: ~{result['estimated_tokens'] - hier['estimated_tokens']:,} tokens")
print(f"Conflicts eliminated: {result['conflicts']} -> {hier['conflicts']}")

---
## 2. Architecture Overview

The supervisor-worker pattern has three layers:

```
                    ┌─────────────┐
                    │  SUPERVISOR  │
                    │  (planner)   │
                    └──────┬──────┘
               ┌───────────┼───────────┐
               │           │           │
        ┌──────▼──┐  ┌────▼────┐  ┌───▼──────┐
        │ Worker A │  │Worker B │  │ Worker C │
        │ research │  │analysis │  │ writing  │
        └──────┬──┘  └────┬────┘  └───┬──────┘
               │           │           │
               └───────────┼───────────┘
                    ┌──────▼──────┐
                    │  AGGREGATOR │
                    │ (merge/rank)│
                    └─────────────┘
```

**Key principles:**
1. **Single point of control** — the supervisor decides what happens and when
2. **Isolated contexts** — workers see only their subtask, not the full conversation
3. **Typed contracts** — every delegation and result follows a schema
4. **Fault isolation** — one worker failing doesn't crash the whole system

In [ ]:
# ── Data contracts for the pattern ───────────────────────────────────────────

class TaskStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"


@dataclass
class SubTask:
    """A unit of work the supervisor delegates to a worker."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    description: str = ""
    assigned_worker: str = ""
    status: TaskStatus = TaskStatus.PENDING
    result: str = ""
    error: str = ""
    started_at: float = 0.0
    completed_at: float = 0.0

    @property
    def duration(self) -> float:
        if self.completed_at and self.started_at:
            return round(self.completed_at - self.started_at, 3)
        return 0.0


@dataclass
class SupervisorPlan:
    """The supervisor's decomposition of a complex task."""
    original_task: str = ""
    subtasks: list = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())


# Quick demo
demo_task = SubTask(
    description="Research EV market trends in 2025",
    assigned_worker="researcher",
)
print(f"SubTask: {demo_task.id}")
print(f"  Description: {demo_task.description}")
print(f"  Worker: {demo_task.assigned_worker}")
print(f"  Status: {demo_task.status.value}")

---
## 3. Designing the Supervisor

The supervisor is responsible for:
1. **Understanding** the high-level task
2. **Decomposing** it into subtasks matched to available workers
3. **Dispatching** subtasks (sequentially or in parallel)
4. **Monitoring** progress and handling failures
5. **Aggregating** results into a final output

We use a mock LLM throughout so every cell runs without API keys.

In [ ]:
# ── Mock LLM for deterministic, offline execution ────────────────────────────

class MockLLM:
    """Simulates LLM responses for the supervisor-worker pattern.
    Replace with real OpenAI calls for production use."""

    def __init__(self, latency: float = 0.1):
        self.latency = latency
        self.call_count = 0

    def complete(self, system_prompt: str, user_prompt: str) -> str:
        """Synchronous mock completion."""
        self.call_count += 1
        time.sleep(self.latency)

        # Route to the right mock based on system prompt keywords
        if "decompose" in system_prompt.lower() or "plan" in system_prompt.lower():
            return self._mock_decomposition(user_prompt)
        elif "research" in system_prompt.lower():
            return self._mock_research(user_prompt)
        elif "analy" in system_prompt.lower():
            return self._mock_analysis(user_prompt)
        elif "writ" in system_prompt.lower():
            return self._mock_writing(user_prompt)
        elif "aggregate" in system_prompt.lower() or "merge" in system_prompt.lower():
            return self._mock_aggregation(user_prompt)
        else:
            return f"Mock response for: {user_prompt[:80]}..."

    async def acomplete(self, system_prompt: str, user_prompt: str) -> str:
        """Async mock completion — same logic, non-blocking sleep."""
        self.call_count += 1
        await asyncio.sleep(self.latency)
        # Reuse sync routing
        if "decompose" in system_prompt.lower() or "plan" in system_prompt.lower():
            return self._mock_decomposition(user_prompt)
        elif "research" in system_prompt.lower():
            return self._mock_research(user_prompt)
        elif "analy" in system_prompt.lower():
            return self._mock_analysis(user_prompt)
        elif "writ" in system_prompt.lower():
            return self._mock_writing(user_prompt)
        elif "aggregate" in system_prompt.lower() or "merge" in system_prompt.lower():
            return self._mock_aggregation(user_prompt)
        else:
            return f"Mock response for: {user_prompt[:80]}..."

    def _mock_decomposition(self, task: str) -> str:
        return json.dumps([
            {"subtask": "Research current market data and trends", "worker": "researcher"},
            {"subtask": "Analyze competitive landscape and key metrics", "worker": "analyst"},
            {"subtask": "Write executive summary and recommendations", "worker": "writer"},
        ])

    def _mock_research(self, task: str) -> str:
        return (
            "Research findings: The global EV market reached $500B in 2025, "
            "growing 25% YoY. Key players: Tesla (18% share), BYD (16%), "
            "Volkswagen (12%). Battery costs fell to $100/kWh. "
            "Government subsidies remain strong in EU and China."
        )

    def _mock_analysis(self, task: str) -> str:
        return (
            "Analysis: Market concentration is moderate (HHI=1,240). "
            "Growth is decelerating from 35% (2024) to 25% (2025). "
            "Margin pressure increasing as subsidies phase out. "
            "Opportunity: solid-state batteries could disrupt by 2027."
        )

    def _mock_writing(self, task: str) -> str:
        return (
            "Executive Summary: The EV market continues robust growth at $500B, "
            "though the rate is moderating. Companies should invest in battery "
            "technology R&D and prepare for subsidy reduction. "
            "Recommendation: Focus on solid-state battery partnerships."
        )

    def _mock_aggregation(self, results: str) -> str:
        return (
            "# EV Market Research Report\n\n"
            "## Market Overview\n"
            "The global EV market reached $500B in 2025 (25% YoY growth).\n\n"
            "## Competitive Analysis\n"
            "Moderate concentration with Tesla, BYD, and VW leading.\n\n"
            "## Recommendations\n"
            "1. Invest in solid-state battery partnerships\n"
            "2. Prepare for subsidy phase-out\n"
            "3. Target emerging markets for growth\n"
        )


llm = MockLLM(latency=0.05)
test = llm.complete("You are a researcher.", "Find EV trends.")
print("Mock LLM test:", test[:80], "...")
print(f"LLM calls so far: {llm.call_count}")

In [ ]:
# ── The Supervisor class ─────────────────────────────────────────────────────

class Supervisor:
    """Orchestrates workers to complete complex tasks.

    Responsibilities:
      - Decompose a task into subtasks
      - Assign subtasks to registered workers
      - Monitor execution and handle failures
      - Aggregate results into a final output
    """

    def __init__(self, llm: MockLLM):
        self.llm = llm
        self.workers: dict[str, "Worker"] = {}
        self.execution_log: list[dict] = []

    def register_worker(self, worker: "Worker") -> None:
        """Add a worker to the available pool."""
        self.workers[worker.name] = worker
        self._log("register", f"Registered worker: {worker.name} ({worker.specialty})")

    def decompose_task(self, task: str) -> SupervisorPlan:
        """Use the LLM to break a task into subtasks."""
        system = (
            "You are a task planner. Decompose the user's task into subtasks. "
            f"Available workers: {list(self.workers.keys())}. "
            "Return a JSON array of objects with 'subtask' and 'worker' keys."
        )
        raw = self.llm.complete(system, task)
        items = json.loads(raw)

        plan = SupervisorPlan(original_task=task)
        for item in items:
            worker_name = item["worker"]
            if worker_name not in self.workers:
                self._log("warning", f"Unknown worker '{worker_name}', skipping.")
                continue
            st = SubTask(
                description=item["subtask"],
                assigned_worker=worker_name,
            )
            plan.subtasks.append(st)

        self._log("plan", f"Created {len(plan.subtasks)} subtasks for: {task[:60]}")
        return plan

    def execute_plan(self, plan: SupervisorPlan) -> list[SubTask]:
        """Execute all subtasks sequentially."""
        for st in plan.subtasks:
            worker = self.workers[st.assigned_worker]
            st.status = TaskStatus.RUNNING
            st.started_at = time.time()
            self._log("dispatch", f"Dispatching to {worker.name}: {st.description[:50]}")

            try:
                st.result = worker.execute(st.description)
                st.status = TaskStatus.COMPLETED
            except Exception as e:
                st.status = TaskStatus.FAILED
                st.error = str(e)
                self._log("error", f"Worker {worker.name} failed: {e}")
            finally:
                st.completed_at = time.time()

        return plan.subtasks

    def aggregate_results(self, subtasks: list[SubTask]) -> str:
        """Merge all worker outputs into a final result."""
        completed = [st for st in subtasks if st.status == TaskStatus.COMPLETED]
        if not completed:
            return "Error: No subtasks completed successfully."

        combined = "\n\n".join(
            f"[{st.assigned_worker}] {st.result}" for st in completed
        )
        system = "You are an aggregator. Merge the following worker outputs into a coherent final report."
        final = self.llm.complete(system, combined)
        self._log("aggregate", f"Aggregated {len(completed)} results.")
        return final

    def run(self, task: str) -> str:
        """Full pipeline: decompose -> execute -> aggregate."""
        plan = self.decompose_task(task)
        results = self.execute_plan(plan)
        return self.aggregate_results(results)

    def _log(self, event: str, message: str) -> None:
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": event,
            "message": message,
        }
        self.execution_log.append(entry)

    def print_log(self) -> None:
        for entry in self.execution_log:
            print(f"  [{entry['event']:>10s}] {entry['message']}")


print("Supervisor class defined.")

---
## 4. Building Worker Agents

Each worker is a **specialized agent** with:
- A **name** and **specialty** description
- Its own **system prompt** (isolated context)
- An `execute(task)` method that returns a string result

Workers never see each other's outputs — only the supervisor does.

In [ ]:
# ── Worker Agent ──────────────────────────────────────────────────────────────

class Worker:
    """A specialized agent that handles one type of subtask."""

    def __init__(self, name: str, specialty: str, system_prompt: str, llm: MockLLM):
        self.name = name
        self.specialty = specialty
        self.system_prompt = system_prompt
        self.llm = llm
        self.task_history: list[dict] = []

    def execute(self, task: str) -> str:
        """Execute a subtask and return the result."""
        result = self.llm.complete(self.system_prompt, task)
        self.task_history.append({
            "task": task,
            "result": result,
            "timestamp": datetime.now().isoformat(),
        })
        return result

    async def aexecute(self, task: str) -> str:
        """Async version for parallel execution."""
        result = await self.llm.acomplete(self.system_prompt, task)
        self.task_history.append({
            "task": task,
            "result": result,
            "timestamp": datetime.now().isoformat(),
        })
        return result

    def __repr__(self) -> str:
        return f"Worker(name={self.name!r}, specialty={self.specialty!r})"


print("Worker class defined.")

In [ ]:
# ── Create three specialized workers ─────────────────────────────────────────

llm = MockLLM(latency=0.05)

researcher = Worker(
    name="researcher",
    specialty="Gathering facts, data, and sources",
    system_prompt=(
        "You are a research specialist. Gather relevant facts, statistics, "
        "and sources for the given topic. Be thorough and cite data points."
    ),
    llm=llm,
)

analyst = Worker(
    name="analyst",
    specialty="Data analysis and competitive insights",
    system_prompt=(
        "You are a data analyst. Analyze the given topic with quantitative "
        "metrics, market comparisons, and trend identification."
    ),
    llm=llm,
)

writer = Worker(
    name="writer",
    specialty="Clear, executive-level writing",
    system_prompt=(
        "You are a business writer. Write clear, concise executive summaries "
        "and actionable recommendations based on the given brief."
    ),
    llm=llm,
)

print("Created workers:")
for w in [researcher, analyst, writer]:
    print(f"  {w}")

In [ ]:
# ── Test each worker independently ───────────────────────────────────────────

print("Researcher output:")
print(textwrap.fill(researcher.execute("EV market trends 2025"), width=80))
print()

print("Analyst output:")
print(textwrap.fill(analyst.execute("EV competitive landscape"), width=80))
print()

print("Writer output:")
print(textwrap.fill(writer.execute("Write executive summary for EV report"), width=80))

---
## 5. Task Decomposition

The supervisor's first job is **planning**: breaking a complex request into discrete, assignable subtasks. This is where the pattern shines — the LLM acts as a reasoning engine that matches subtasks to worker specialties.

Good decomposition has these properties:
- **Completeness** — all aspects of the task are covered
- **Independence** — subtasks can run in parallel where possible
- **Right-sized** — neither too granular nor too broad

In [ ]:
# ── Wire up supervisor with workers ──────────────────────────────────────────

supervisor = Supervisor(llm=llm)
supervisor.register_worker(researcher)
supervisor.register_worker(analyst)
supervisor.register_worker(writer)

print(f"Supervisor has {len(supervisor.workers)} workers: {list(supervisor.workers.keys())}")

In [ ]:
# ── Decompose a complex task ─────────────────────────────────────────────────

task = "Create a comprehensive market research report on electric vehicle adoption trends, competitive landscape, and strategic recommendations for a traditional automaker considering an EV transition."

plan = supervisor.decompose_task(task)

print(f"Original task: {plan.original_task[:80]}...")
print(f"Decomposed into {len(plan.subtasks)} subtasks:\n")

for i, st in enumerate(plan.subtasks, 1):
    print(f"  {i}. [{st.assigned_worker:>10s}] {st.description}")
    print(f"     Status: {st.status.value} | ID: {st.id}")
    print()

In [ ]:
# ── Examine the plan structure ───────────────────────────────────────────────

print("Plan metadata:")
print(f"  Created at:  {plan.created_at}")
print(f"  Subtask IDs: {[st.id for st in plan.subtasks]}")
print(f"  Workers used: {set(st.assigned_worker for st in plan.subtasks)}")
print()

# Verify all workers exist
unmatched = [
    st.assigned_worker for st in plan.subtasks
    if st.assigned_worker not in supervisor.workers
]
if unmatched:
    print(f"WARNING: Unmatched workers: {unmatched}")
else:
    print("All subtasks mapped to registered workers.")

---
## 6. Parallel Execution

Sequential execution is simple but slow. Since workers have **isolated contexts**, we can run independent subtasks in parallel using `asyncio`.

This section compares sequential vs. parallel execution time.

In [ ]:
# ── Sequential execution (baseline) ──────────────────────────────────────────

# Reset the plan subtasks to PENDING
for st in plan.subtasks:
    st.status = TaskStatus.PENDING
    st.result = ""

start = time.time()
sequential_results = supervisor.execute_plan(plan)
sequential_time = time.time() - start

print(f"Sequential execution: {sequential_time:.3f}s")
print()
for st in sequential_results:
    print(f"  [{st.assigned_worker}] {st.status.value} in {st.duration:.3f}s")
    print(f"    Result: {st.result[:70]}...")
    print()

In [ ]:
# ── Parallel execution with asyncio ──────────────────────────────────────────

async def execute_plan_parallel(
    supervisor: Supervisor, plan: SupervisorPlan
) -> list[SubTask]:
    """Run all subtasks concurrently using asyncio.gather."""

    async def run_one(st: SubTask) -> SubTask:
        worker = supervisor.workers[st.assigned_worker]
        st.status = TaskStatus.RUNNING
        st.started_at = time.time()
        try:
            st.result = await worker.aexecute(st.description)
            st.status = TaskStatus.COMPLETED
        except Exception as e:
            st.status = TaskStatus.FAILED
            st.error = str(e)
        finally:
            st.completed_at = time.time()
        return st

    # Reset subtasks
    for st in plan.subtasks:
        st.status = TaskStatus.PENDING
        st.result = ""

    tasks = [run_one(st) for st in plan.subtasks]
    results = await asyncio.gather(*tasks)
    return list(results)


# Run parallel version
start = time.time()
parallel_results = await execute_plan_parallel(supervisor, plan)
parallel_time = time.time() - start

print(f"Parallel execution: {parallel_time:.3f}s")
print()
for st in parallel_results:
    print(f"  [{st.assigned_worker}] {st.status.value} in {st.duration:.3f}s")
    print(f"    Result: {st.result[:70]}...")
    print()

In [ ]:
# ── Compare sequential vs parallel ───────────────────────────────────────────

speedup = sequential_time / parallel_time if parallel_time > 0 else float('inf')

print("Execution Comparison")
print("=" * 40)
print(f"  Sequential: {sequential_time:.3f}s")
print(f"  Parallel:   {parallel_time:.3f}s")
print(f"  Speedup:    {speedup:.1f}x")
print()
print(f"With {len(plan.subtasks)} workers, theoretical max speedup is {len(plan.subtasks)}x.")
print("In practice, speedup depends on the slowest worker (Amdahl's law).")

---
## 7. Result Aggregation

Once workers finish, the supervisor must **merge** their outputs into a coherent whole. Aggregation strategies include:

| Strategy | When to use |
|----------|------------|
| **Concatenation** | Workers produce independent sections (research report) |
| **LLM synthesis** | Outputs overlap or conflict; need a unified narrative |
| **Voting / ranking** | Workers solve the same problem; pick the best answer |
| **Structured merge** | Outputs are JSON/data that can be merged programmatically |

In [ ]:
# ── Strategy 1: Simple concatenation ─────────────────────────────────────────

def aggregate_concat(subtasks: list[SubTask]) -> str:
    """Concatenate all worker outputs with headers."""
    sections = []
    for st in subtasks:
        if st.status == TaskStatus.COMPLETED:
            sections.append(f"## {st.assigned_worker.title()} Output\n{st.result}")
    return "\n\n".join(sections)


concat_result = aggregate_concat(parallel_results)
print("Concatenation aggregation:")
print("-" * 40)
print(concat_result)

In [ ]:
# ── Strategy 2: LLM-based synthesis ──────────────────────────────────────────

def aggregate_llm_synthesis(subtasks: list[SubTask], llm: MockLLM) -> str:
    """Use the LLM to merge worker outputs into a coherent report."""
    completed = [st for st in subtasks if st.status == TaskStatus.COMPLETED]
    if not completed:
        return "No completed subtasks to aggregate."

    combined_input = "\n\n".join(
        f"--- {st.assigned_worker} ---\n{st.result}" for st in completed
    )

    system = (
        "You are an aggregator. Merge the following worker outputs into a single, "
        "well-structured report. Remove redundancy, resolve conflicts, "
        "and ensure a coherent narrative."
    )
    return llm.complete(system, combined_input)


synthesis_result = aggregate_llm_synthesis(parallel_results, llm)
print("LLM synthesis aggregation:")
print("-" * 40)
print(synthesis_result)

In [ ]:
# ── Strategy 3: Structured merge (JSON outputs) ──────────────────────────────

def aggregate_structured(subtasks: list[SubTask]) -> dict:
    """Merge structured outputs into a single data object."""
    report = {
        "title": "EV Market Research Report",
        "generated_at": datetime.now().isoformat(),
        "sections": {},
        "metadata": {
            "total_workers": len(subtasks),
            "completed": sum(1 for st in subtasks if st.status == TaskStatus.COMPLETED),
            "failed": sum(1 for st in subtasks if st.status == TaskStatus.FAILED),
        },
    }
    for st in subtasks:
        if st.status == TaskStatus.COMPLETED:
            report["sections"][st.assigned_worker] = {
                "content": st.result,
                "task": st.description,
                "duration_s": st.duration,
            }
    return report


structured = aggregate_structured(parallel_results)
print("Structured merge:")
print(json.dumps(structured, indent=2)[:600], "...")

---
## 8. Error Handling

In production, workers fail. Networks time out, APIs return errors, LLMs hallucinate invalid JSON. The supervisor must handle all of this gracefully.

Strategies:
1. **Retry with backoff** — transient failures often resolve
2. **Fallback workers** — route to a different worker if the primary fails
3. **Partial results** — deliver what you have, note what's missing
4. **Circuit breaker** — stop retrying after N consecutive failures

In [ ]:
# ── A worker that sometimes fails ────────────────────────────────────────────

class UnreliableWorker(Worker):
    """A worker with a configurable failure rate for testing."""

    def __init__(self, *args, failure_rate: float = 0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.failure_rate = failure_rate
        self.attempt_count = 0

    def execute(self, task: str) -> str:
        self.attempt_count += 1
        if random.random() < self.failure_rate:
            raise ConnectionError(
                f"Worker '{self.name}' failed on attempt {self.attempt_count}: "
                "simulated network timeout"
            )
        return super().execute(task)


unreliable = UnreliableWorker(
    name="flaky_researcher",
    specialty="Research (unreliable)",
    system_prompt="You are a researcher.",
    llm=llm,
    failure_rate=0.6,
)

# Test: try 5 times, observe failures
random.seed(42)
for i in range(5):
    try:
        result = unreliable.execute("Find EV trends")
        print(f"  Attempt {i+1}: SUCCESS - {result[:50]}...")
    except ConnectionError as e:
        print(f"  Attempt {i+1}: FAILED - {e}")

In [ ]:
# ── Retry with exponential backoff ───────────────────────────────────────────

def execute_with_retry(
    worker: Worker,
    task: str,
    max_retries: int = 3,
    base_delay: float = 0.1,
) -> tuple[str, int]:
    """Execute a worker task with retries and exponential backoff.

    Returns (result, attempts_used).
    Raises the last exception if all retries are exhausted.
    """
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            result = worker.execute(task)
            return result, attempt
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                delay = base_delay * (2 ** (attempt - 1))
                print(f"    Retry {attempt}/{max_retries} after {delay:.2f}s: {e}")
                time.sleep(delay)
    raise last_error  # type: ignore


# Test retry mechanism
random.seed(10)
unreliable.failure_rate = 0.5
unreliable.attempt_count = 0

try:
    result, attempts = execute_with_retry(unreliable, "EV market data", max_retries=5)
    print(f"\nSuccess after {attempts} attempt(s): {result[:60]}...")
except Exception as e:
    print(f"\nAll retries exhausted: {e}")

In [ ]:
# ── Supervisor with fault tolerance ──────────────────────────────────────────

class ResilientSupervisor(Supervisor):
    """A supervisor that retries failed workers and handles partial results."""

    def __init__(self, llm: MockLLM, max_retries: int = 3):
        super().__init__(llm)
        self.max_retries = max_retries

    def execute_plan(self, plan: SupervisorPlan) -> list[SubTask]:
        """Execute with retry logic and partial-result tolerance."""
        for st in plan.subtasks:
            worker = self.workers[st.assigned_worker]
            st.status = TaskStatus.RUNNING
            st.started_at = time.time()
            self._log("dispatch", f"Dispatching to {worker.name}: {st.description[:50]}")

            try:
                st.result, attempts = execute_with_retry(
                    worker, st.description, max_retries=self.max_retries
                )
                st.status = TaskStatus.COMPLETED
                self._log("success", f"{worker.name} completed in {attempts} attempt(s)")
            except Exception as e:
                st.status = TaskStatus.FAILED
                st.error = str(e)
                self._log("failure", f"{worker.name} failed after {self.max_retries} retries: {e}")
            finally:
                st.completed_at = time.time()

        return plan.subtasks

    def aggregate_results(self, subtasks: list[SubTask]) -> str:
        """Aggregate with awareness of missing sections."""
        completed = [st for st in subtasks if st.status == TaskStatus.COMPLETED]
        failed = [st for st in subtasks if st.status == TaskStatus.FAILED]

        parts = []
        if completed:
            combined = "\n\n".join(
                f"[{st.assigned_worker}] {st.result}" for st in completed
            )
            system = "You are an aggregator. Merge worker outputs into a coherent report."
            parts.append(self.llm.complete(system, combined))

        if failed:
            parts.append("\n---\nNote: The following sections could not be completed:")
            for st in failed:
                parts.append(f"  - {st.assigned_worker}: {st.error}")

        return "\n".join(parts) if parts else "Error: No results produced."


print("ResilientSupervisor class defined.")

In [ ]:
# ── Test resilient supervisor with a mix of reliable and unreliable workers ──

random.seed(42)

resilient_sup = ResilientSupervisor(llm=llm, max_retries=3)

# Register a mix of workers
resilient_sup.register_worker(researcher)  # reliable
resilient_sup.register_worker(analyst)      # reliable
resilient_sup.register_worker(UnreliableWorker(
    name="writer",
    specialty="Writing (sometimes fails)",
    system_prompt="You are a writer.",
    llm=llm,
    failure_rate=0.4,
))

# Run the full pipeline
final_report = resilient_sup.run(
    "Create an EV market report with research, analysis, and executive summary."
)

print("Final Report:")
print("=" * 60)
print(final_report)
print()
print("Execution log:")
resilient_sup.print_log()

In [ ]:
# ── Circuit breaker pattern ──────────────────────────────────────────────────

class CircuitBreaker:
    """Stops calling a worker after N consecutive failures.

    States:
      CLOSED  — normal operation, calls pass through
      OPEN    — worker is considered down, calls fail immediately
      HALF_OPEN — allow one test call to see if worker recovered
    """

    def __init__(self, failure_threshold: int = 3, reset_timeout: float = 5.0):
        self.failure_threshold = failure_threshold
        self.reset_timeout = reset_timeout
        self.failure_count = 0
        self.state = "CLOSED"
        self.last_failure_time = 0.0

    def call(self, func: Callable, *args, **kwargs) -> Any:
        if self.state == "OPEN":
            if time.time() - self.last_failure_time > self.reset_timeout:
                self.state = "HALF_OPEN"
            else:
                raise RuntimeError("Circuit breaker is OPEN — call rejected.")

        try:
            result = func(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise e

    def _on_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def _on_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"


# Demo the circuit breaker
random.seed(0)
cb = CircuitBreaker(failure_threshold=3, reset_timeout=1.0)
flaky = UnreliableWorker(
    name="very_flaky", specialty="test",
    system_prompt="You are a researcher.", llm=llm, failure_rate=0.8,
)

for i in range(6):
    try:
        result = cb.call(flaky.execute, "test task")
        print(f"  Call {i+1}: SUCCESS (state={cb.state})")
    except RuntimeError as e:
        print(f"  Call {i+1}: BLOCKED - {e}")
    except ConnectionError as e:
        print(f"  Call {i+1}: FAILED  (state={cb.state}, failures={cb.failure_count})")

---
## Full End-to-End Run

Let's put it all together: supervisor decomposes, workers execute in parallel, results are aggregated.

In [ ]:
# ── End-to-end: full supervisor-worker pipeline ──────────────────────────────

llm_fresh = MockLLM(latency=0.1)

# Build the team
sup = Supervisor(llm=llm_fresh)
sup.register_worker(Worker(
    "researcher", "Market research",
    "You are a research specialist.", llm_fresh,
))
sup.register_worker(Worker(
    "analyst", "Data analysis",
    "You are a data analyst.", llm_fresh,
))
sup.register_worker(Worker(
    "writer", "Business writing",
    "You are a business writer.", llm_fresh,
))

# Run
task = (
    "Produce a market intelligence brief on the state of generative AI adoption "
    "in Fortune 500 companies, including market sizing, key vendors, and risks."
)

start = time.time()
report = sup.run(task)
elapsed = time.time() - start

print("FINAL REPORT")
print("=" * 60)
print(report)
print()
print(f"Total time: {elapsed:.2f}s | LLM calls: {llm_fresh.call_count}")

In [ ]:
# ── Execution log review ─────────────────────────────────────────────────────

print("Supervisor Execution Log")
print("=" * 60)
sup.print_log()
print()
print(f"Total log entries: {len(sup.execution_log)}")

---
## LangGraph Implementation (Reference)

The pattern above is framework-agnostic. Here is how you would express the same idea using **LangGraph's** `StateGraph`. This cell is for reference — it uses the mock LLM so it runs without an API key.

In [ ]:
# ── LangGraph-style state graph (conceptual, uses mocks) ─────────────────────

from typing import TypedDict

try:
    from langgraph.graph import StateGraph, END
    LANGGRAPH_AVAILABLE = True
except ImportError:
    LANGGRAPH_AVAILABLE = False
    print("langgraph not installed — showing conceptual code only.")


class ResearchState(TypedDict):
    task: str
    plan: list[dict]
    research: str
    analysis: str
    draft: str
    final_report: str


def supervisor_node(state: ResearchState) -> dict:
    """Supervisor decomposes the task."""
    plan = json.loads(llm_fresh.complete(
        "You are a task planner. Decompose the task.",
        state["task"],
    ))
    return {"plan": plan}


def research_node(state: ResearchState) -> dict:
    result = llm_fresh.complete("You are a researcher.", state["task"])
    return {"research": result}


def analysis_node(state: ResearchState) -> dict:
    result = llm_fresh.complete("You are an analyst.", state["task"])
    return {"analysis": result}


def writing_node(state: ResearchState) -> dict:
    result = llm_fresh.complete("You are a writer.", state["task"])
    return {"draft": result}


def aggregator_node(state: ResearchState) -> dict:
    combined = f"Research: {state['research']}\nAnalysis: {state['analysis']}\nDraft: {state['draft']}"
    result = llm_fresh.complete("You are an aggregator. Merge these outputs.", combined)
    return {"final_report": result}


if LANGGRAPH_AVAILABLE:
    graph = StateGraph(ResearchState)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("researcher", research_node)
    graph.add_node("analyst", analysis_node)
    graph.add_node("writer", writing_node)
    graph.add_node("aggregator", aggregator_node)

    graph.set_entry_point("supervisor")
    graph.add_edge("supervisor", "researcher")
    graph.add_edge("supervisor", "analyst")
    graph.add_edge("supervisor", "writer")
    graph.add_edge("researcher", "aggregator")
    graph.add_edge("analyst", "aggregator")
    graph.add_edge("writer", "aggregator")
    graph.add_edge("aggregator", END)

    app = graph.compile()
    result = app.invoke({
        "task": "Analyze the AI chip market",
        "plan": [], "research": "", "analysis": "", "draft": "", "final_report": "",
    })
    print("LangGraph result:")
    print(result["final_report"])
else:
    print("Install langgraph to run this cell: pip install langgraph")
    print("The conceptual code above shows the StateGraph wiring.")

---
## 9. Exercises

### Exercise 1 — Conceptual (No Code)

You are building a customer-support system with these agents: **triage**, **billing**, **technical**, **escalation**, **satisfaction-survey**.

1. Draw the supervisor-worker hierarchy. Which agent is the supervisor? Which are workers?
2. Should `escalation` be a worker or a **second-level supervisor** with its own workers? Justify.
3. A billing worker fails mid-conversation. Describe three recovery strategies the supervisor could use, and pick the best one for a customer-facing system.

---

### Exercise 2 — Coding: Build a Research Supervisor

Extend the `Supervisor` class to build a **research supervisor** that:

1. Accepts a research question (e.g., "What are the environmental impacts of lithium mining?")
2. Decomposes it into **at least 4 subtasks** assigned to different workers:
   - `source_finder` — identifies relevant papers / articles
   - `data_extractor` — pulls key statistics
   - `critic` — identifies weaknesses or bias in sources
   - `synthesizer` — writes the final research brief
3. Runs the first three workers in **parallel**, then feeds their outputs to the `synthesizer`
4. Includes retry logic for the `source_finder` (external APIs are flaky)

Starter code is below. Fill in the `TODO` sections.

In [ ]:
# ── Exercise 2: Starter code ─────────────────────────────────────────────────

class ResearchSupervisor(Supervisor):
    """A supervisor specialized for research tasks."""

    def __init__(self, llm: MockLLM):
        super().__init__(llm)
        self._setup_workers()

    def _setup_workers(self):
        """Register the four specialized workers."""
        # TODO: Create and register these workers:
        #   - source_finder
        #   - data_extractor
        #   - critic
        #   - synthesizer
        pass

    async def run_research(self, question: str) -> str:
        """Full research pipeline: decompose -> parallel workers -> synthesize."""
        # TODO: Implement the pipeline:
        #   1. Decompose the question into subtasks
        #   2. Run source_finder, data_extractor, and critic in parallel
        #   3. Feed their results to the synthesizer
        #   4. Return the final research brief
        pass


# ── Test your implementation ──────────────────────────────────────────────────
# Uncomment once you've filled in the TODOs:

# research_sup = ResearchSupervisor(llm=MockLLM(latency=0.05))
# brief = await research_sup.run_research(
#     "What are the environmental impacts of lithium mining?"
# )
# print(brief)

### Exercise 3 — Design: Content Generation Pipeline

You are designing a supervisor-worker system for an **automated content generation pipeline** that produces weekly industry newsletters. The pipeline must:

1. Scan 50+ RSS feeds for relevant articles
2. Summarize the top 10 articles
3. Write an editorial connecting the themes
4. Generate a subject line and preview text
5. Format for email (HTML) and social media (short posts)

**Design tasks:**

1. **Worker inventory:** List every worker you would create, with its name, specialty, and inputs/outputs.
2. **Dependency graph:** Which workers depend on others? Draw the execution order.
3. **Parallelism plan:** Which workers can run in parallel? What is the critical path?
4. **Failure modes:** For each worker, what is the most likely failure? What is the supervisor's recovery strategy?
5. **Scaling:** If you need to process 500 RSS feeds instead of 50, what changes? Would you add a second level of supervisors?

Write your design in the cell below as markdown or code comments.

In [ ]:
# ── Exercise 3: Write your design here ───────────────────────────────────────
#
# Worker inventory:
#   1. ...
#   2. ...
#
# Dependency graph:
#   ...
#
# Parallelism plan:
#   ...
#
# Failure modes:
#   ...
#
# Scaling strategy:
#   ...
#

---

## Key Takeaways

| Concept | What you learned |
|---------|------------------|
| **Hierarchy beats flat** | Supervisor-worker scales where group chat does not |
| **Isolated contexts** | Workers see only their subtask — cheaper and safer |
| **Task decomposition** | The supervisor's planning step is the highest-leverage LLM call |
| **Parallel execution** | Independent subtasks should run concurrently via asyncio |
| **Aggregation strategies** | Concatenation, LLM synthesis, structured merge — pick the right one |
| **Fault tolerance** | Retry, circuit breaker, and partial results keep production systems running |

**Next chapter:** Chapter 11 explores the **Evaluator-Optimizer** pattern — a supervisor that doesn't just delegate, but iteratively improves worker outputs through feedback loops.